In [8]:
import paarti.utils.maos_utils as mu
import paarti.utils.koa_utils as ku
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from pathlib import Path
from astropy.io import fits

In [11]:
ku.fetch_koa(date='2014-08-02', koa_dir=Path("/Users/bdigia/myg3/data/"), verbose=True)

submitting request...
Result downloaded to file [/Users/bdigia/myg3/datanirc2_search_2014-08-02.tbl]
        koaid          instrume  targname ... slitmm  slitname slsname
                                          ...   mm                    
---------------------- -------- --------- ... ------ --------- -------
N2.20140802.01496.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.01535.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.01570.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.01650.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.02406.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.02444.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.02479.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.02514.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.02555.fits    NIRC2        -- ...    0.0      none   clear
N2.20140802.02595.fits    NIRC2        -- ...  

In [ ]:
# Grab KOA image for which to run a MAOS sim
koa_file = Path("/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.03304.fits")
old_hdr, clean_img = ku.clean_koa(koa_file)

In [ ]:
# Compute metrics on cleaned KOA image 
# re-defining koa_file as string because calc_strehl_on_sky does not
# iterate over Path object correctly in parsing
koa_file = ["/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.03304.fits", 
            "/Users/bdigia/myg3/data/KOA_89772/NIRC2/raw/cal/N2.20160803.04422.fits"]
koa_strehls, koa_fwhms, koa_rmswfes = mu.calc_strehl_on_sky(koa_file, "temp.txt")

In [ ]:
# Simulation seeds
seeds = np.array([1, 1000, 5000, 10000])
# Simulation types/modes
simtypes = ['piston', 'psd+ncpa-unseen', 'psd+ncpa-seen']
baseroot = Path("/Users/bdigia/work/ao/keck/maos/keck/my_base/")
# Run simulation for each KOA image in koa_file array
for koa in koa_file:
    print(koa)
    with fits.open(koa) as koa_fits:
        hdu = koa_fits[0]
        hdr = hdu.header
        # Zenith angle
        angle = np.degrees(np.arccos(1.0/float(hdr['AIRMASS'])))
        # Calculate atm parameters
        koa_path = Path(koa)
        fried, turbpro, windspds, winddrcts, _, _, _, _ = mu.estimate_on_sky_conditions(koa, koa_path.parent.as_posix() + "/", verbose=True)

        for seed in seeds:
            for mode in simtypes:
                if mode == 'piston':
                    surf_cmd = ["Keck_ncpa_rmswfe130nm.fits"]
                    # Fetch name of current input PSD FITS file in MAOS config file keck_sim.conf
                    psd_file = ''
                elif mode == 'psd+ncpa-seen':
                    mode = 'surf_wfs1'
                    surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=1; SURFEVL=1; seed=10;'"]
                    psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
                elif mode == 'psd+ncpa-unseen':
                    mode = 'surf_wfs0'
                    surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=0; SURFEVL=1; seed=10;'"]
                    psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
                else:
                    raise ValueError(f"Invalid MAOS simulation type '{type}'. Valid types are currently: 'piston', 'psd+ncpa-seen', 'psd+ncpa-unseen'. See help() for further info")
        
                # Must be in MAOS simulation directory to run successfully
                if os.getcwd() != baseroot.as_posix():
                    print("Moving current working directory to MAOS simulation directory...\n")
                    os.chdir(baseroot)

                maos_cmd = f"maos -o A_keck_scao_lgs_koa_{mode}_comp_{koa[60:65]}_seed{seed}_epoch{koa[51:59]} -c A_keck_scao_lgs.conf sim.seeds={seed} sim.zadeg={angle} sim.wspsd={psd_file} atm.r0z={fried} atm.wt={turbpro} atm.ws={windspds} atm.wddeg={winddrcts} surf={surf_cmd} -O"
                os.system(maos_cmd)